In [1]:
import random
import math
import hashlib
import struct
from typing import Callable, Any
import numpy as np

In [11]:
class IntegerHash:
    def __init__(self):
        self._seed = random.randint(0, 2**32 - 1)

    def __call__(self, item: Any) -> int:
        text = str(item) + str(self._seed)
        hash_bytes = hashlib.md5(text.encode('utf-8')).digest()
        return struct.unpack('>I', hash_bytes[:4])[0]

In [3]:
class HLLSketch:
    def __init__(self, b: int, hashf: Callable[[Any], int]):
        self._b = b
        self._m = 1 << b
        self._hash = hashf
        self._registers = [0] * self._m

        self._alpha_hl = 0.39701
        self._alpha_hll = HLLSketch.get_alpha_hll(self._m)

    def add(self, item: Any):
        h = self._hash(item)
        j = h & (self._m - 1)
        w = h >> self._b
        self._registers[j] = max(self._registers[j], self._rho(w))

    def estimate_hyperlog(self) -> float:
        """
        Arithmetic Mean
        """
        arithmetic_mean = sum(self._registers) / self._m
        return self._alpha_hl * self._m * (2 ** arithmetic_mean)

    def estimate_hyperloglog(self) -> float:
        """
        Harmonic Mean
        """
        sum_inverse_pow2 = sum(2 ** (-r) for r in self._registers)
        if sum_inverse_pow2 == 0:
            return 0.0

        return self._alpha_hll * (self._m ** 2) / sum_inverse_pow2

    def n_empty_registers(self) -> int:
        return self._registers.count(0)

    def _rho(self, w: int) -> int:
        if w == 0:
            return 32 - self._b + 1
        return (w & -w).bit_length()

    @staticmethod
    def get_alpha_hll(m: int) -> float:
        if m == 16:
            return 0.673
        elif m == 32:
            return 0.697
        elif m == 64:
            return 0.709
        else:
            return 0.7213 / (1 + 1.079 / m)

In [ ]:
b = 10
k = 1 << b
hashf = IntegerHash()

boundary = k * math.log(k)
test_cardinalities = [
    int(boundary * 0.25),
    int(boundary * 0.5),
    int(boundary * 0.9),
    int(boundary),
    int(boundary * 1.1),
    int(boundary * 2.0),
    int(boundary * 4.0)
]

print(f"Number of Registers: {k}")
print(f"Theoretical Boundry: ~ {boundary:.2f}")
print("-" * 80)
print(f"{'True N':<10} | {'HyperLog':<12} {'Err(%)':<8} | {'HyperLogLog':<12} {'Err(%)':<8} | {'Empty Registers':<10}")
print("-" * 80)

for n in test_cardinalities:
    est_hl_values = []
    err_ll_values = []
    est_hll_values = []
    err_hll_values = []
    zeros_values = []

    for k in range(100):
        sketch = HLLSketch(b=b, hashf=hashf)

        offset = random.randint(0, 1000000)
        stream = list(range(offset, offset + n))
        random.shuffle(stream)
        for item in stream:
            sketch.add(item)

        est_hl = sketch.estimate_hyperlog()
        est_hll = sketch.estimate_hyperloglog()

        est_hl_values += [est_hl]
        est_hll_values += [est_hll]

        err_ll_values += [abs(est_hl - n) / n * 100]
        err_hll_values += [abs(est_hll - n) / n * 100]
        zeros_values += [sketch.n_empty_registers()]

    print(f"{n:<10} | {np.mean(est_hl_values):<12.2f} {np.mean(err_ll_values):<8.2f} | {np.mean(est_hll_values):<12.2f} {np.mean(err_hll_values):<8.2f} | {np.mean(zeros_values):<10}")

print("-" * 80)

Number of Registers: 1024
Theoretical Boundry: ~ 7097.83
--------------------------------------------------------------------------------
True N     | HyperLog     Err(%)   | HyperLogLog  Err(%)   | Empty Registers
--------------------------------------------------------------------------------
1774       | 1814.20      3.15     | 1927.32      8.64     | 180.73    
3548       | 3553.06      2.57     | 3571.14      2.05     | 31.62     
6388       | 6379.41      2.73     | 6380.73      2.55     | 2.09      
7097       | 7132.52      2.81     | 7111.00      2.37     | 1.11      
7097       | 7133.73      2.62     | 7124.99      2.56     | 0.95      
14195      | 14198.51     2.31     | 14195.78     1.97     | 0.0       
28391      | 28259.69     2.76     | 28221.99     2.21     | 0.0       
--------------------------------------------------------------------------------
